### --- Day 8: Playground ---

Jeremy Bloom solutions for  [Advent of Code](https://adventofcode.com): [2025 Day 8](https://adventofcode.com/2025/day/8)


In [563]:
def get_input_file(filename):
  try:
    with open(filename, 'r') as f:
      content = f.read()
      #content = content.strip() # disable for day 6 as i want to  keep trailing spaces on each line
      contents_as_list = content.split("\n")
      if contents_as_list[-1] == "":
        contents_as_list = contents_as_list[:-1]
  except FileNotFoundError:
    print("no such file found")
  return contents_as_list

day8_example = get_input_file("day8_example.txt")

In [564]:
day8_example

['162,817,812',
 '57,618,57',
 '906,360,560',
 '592,479,940',
 '352,342,300',
 '466,668,158',
 '542,29,236',
 '431,825,988',
 '739,650,466',
 '52,470,668',
 '216,146,977',
 '819,987,18',
 '117,168,530',
 '805,96,715',
 '346,949,466',
 '970,615,88',
 '941,993,340',
 '862,61,35',
 '984,92,344',
 '425,690,689']

In [566]:
class JunctionBox():
  '''
  Track x,y,z and power status
  supports get_distance_from() 
  does NOT track connnections or sets
  '''
  def __init__(self, xyz_in):
    xyz = xyz_in.split(",")
    assert len(xyz) ==3, "error - invalid junction box"
    self.x = int(xyz[0])
    self.y = int(xyz[1])
    self.z = int(xyz[2])
    self.power = 0

  def __repr__(self):
    s = f"{self.x},{self.y},{self.z}"
    return s

  def stringify(self):
    return f"{self.x},{self.y},{self.z}"
  
  def getx(self):
    return self.x
  def gety(self):
    return self.y
  def getz(self):
    return self.z
  
  def get_distance_from(self, other_box):
    x_diff = self.x - other_box.getx()
    y_diff = self.y - other_box.gety()
    z_diff = self.z - other_box.getz()
    n = x_diff**2 + y_diff**2 + z_diff**2
    return n ** .5

  def is_powered(self):
    return self.power

  def set_to_powered(self):
    self.power = 1
    
x = JunctionBox('57,618,57')
x
x.is_powered()
x.set_to_powered()
x.is_powered()

1

In [497]:
if 0:
  X = []
  for row in day8_example:
    X.append( JunctionBox(row) )
  print(X)
  X[0].get_distance_from( X[1] )

In [851]:
class Playground():
  '''
  Manages the JunctionBoxes and Logical Circuits.
  - JunctionBoxes are int from 0..X
  - a circuit is one or more JBs connected directly or indirectly
  - when connecting JBs, we are connecting the two circuits into a single circuit
  
  JunctionBoxes managed via
    self.boxes[] - the list of JunctionBoxes (of which each tracks x,y,z and power status
    cmap[] - list of lists with direct connectivity mapping.  [i][j] = 1 means box i connects directly to box j (and vv)
           - indirect connections can be traced, or can be seen in Logical Circuits

  Logical Circuits managed via
  - Tracks the circuits and, by id only, the associated Junction Boxes
  - circuits consist of one or more boxes stored as ints (0.. num_boxes).
  - circuit is defined by a) its membership and b) it's lowest box/int val which is the circuit name
  - so given a circuit named i it must contain box i and can include zero or more ints of i+1 or above
  - if a box of b < i is added to the above circuit, circuit b is created with circuit b + circuit i contents, and circuit i is emptied
  - status - 1 if the logical circuit is active / non empty
  - parent - if status: 0, this is a pointer to the circuit that was merged with this.  By def a lower number [i].  
           - Note multiple merges may take place so have to follow the chain
  - members - if status: 1, members of this circuit not counting [i] itself (which is in idmins[i]  
  - init has every circuit [i] having it's i value in idmins and empty list in members
  - if circuits [3] and [5] are merged:
     - circuit[5].status set to 0
     - circuit[3].append(5)
     - this means circuit 3 has box 3 and box 5, and there is no longer a circuit 5


  find_two_closest_unconnected_boxes()
  - looks though all JunctionBoxes to find the two closest boxes that aren't a) the same box b) already directly connected to each other
  - returns best_i, best_j


  add_val(i,j)
    - key function that adds circuit i and j together.  
    - The lower int (i,j) circuit is updated to include the other circuit.
    - The higher int circuit is marked as acitve=0 - it is no longer a circuit
    - for tracing, the higher int circuit idmin[y] is set to new parent circuit (i or j)

  

  '''
  
  def __init__(self, data_in):
    self.num_boxes = len(data_in)
    self.status = []      # if status[i] == 1:  circuit i has the name i
    self.members = []     # if status[i] == 1, members[i] contains the members of circuit i (except for i itself)
    self.parent = []      # if status[i] == 0, if (status = 0 and i = parent[j]) then j is in i or a later parent(s)
    self.boxes = []       # these are the box objects, with xyx value and power status

    self.term_condition = 10    # if a positive integer, the number of physical connections to terminate at
    self.term_condition = 0     # if zero, run until there is only a single virtual circuit
    
    self.last_i = -1 # box id that was last connected
    self.last_j = -1 # box id that was last connected
    
    for i in range(self.num_boxes):
      self.boxes.append( JunctionBox( data_in[i] ) )
      self.status.append( 1 )
      self.members.append( [] )
      self.parent.append( -1 )
      
    # mapping tracks the individual connections, in case I need these later
    self.cmap = [] # 2d matrix of connections i:j
    for i in range(self.num_boxes):
      this_i_cmap = []
      for j in range(self.num_boxes):
        if i != j:
          this_i_cmap.append(0) # not connected
        else:
          this_i_cmap.append(1) # connected
      self.cmap.append(this_i_cmap)

    # setup distance cache 
    self.d_cache = []
    for i in range(self.num_boxes):
      this_row = []
      for j in range(self.num_boxes):
        this_row.append(-1)
      self.d_cache.append(this_row)
      
  def print_distance_cache(self):
    s = "d_c: "
    for i in range(self.num_boxes):
      s +=  f"  {i:3d} "
    print(s)
    
    for i in range(self.num_boxes):
      s = f" {i:4d}: "
      for j in range(self.num_boxes):
        s += f" {self.d_cache[i][j]:4.0f} "
      print(s)

  def get_box(self, i):
    return self.boxes[i]
  
  def get_box_report(self, i):
    o = self.boxes[i]
    s = f"{o.getx()},{o.gety()},{o.getz()}"
    if o.is_powered():
      return s + " is powered"
    else:
      return s + " (unpowered)"
  
  def get_box_direct_connections(self, i):
    # physical connections
    o = []
    for x in self.cmap[i]:
      o.append(x)
    return o

  def print_box_connection_map(self):
    # physical connections
    s = "cmap: "
    for i in range(self.num_boxes):
      s +=  f" {i:3d} "
    print(s)
    
    for i in range(self.num_boxes):
      s = f"{i:4d}: "
      for j in range(self.num_boxes):
        s += f" {self.cmap[i][j]:3d} "
      print(s)
      

  def are_boxes_directly_connected(self, i, j):
    return self.cmap[i][j]

  
  def get_num_unpowered(self):
    # looks at JunctionBoxes
    num_powered = 0
    num_unpowered = 0
    for i in range(self.num_boxes):
      if self.boxes[i].is_powered():
        num_powered += 1
      else:
        num_unpowered += 1
    return num_unpowered

    
  def find_two_closest_unconnected_boxes(self, debug = 0):
    """
    Looks through all JunctionBoxes to find the two closest JunctionBoxes that are not already connected to each other.
    These can be any combination of powered or unpowered JunctionBoxes.  
    
    (so it will connection powered to powered even if in same circuit! this won't change the logical circuits, but is in the example)
    
    """

    # # scan to make sure at least one box is unpowered - otherwise there is nothing to find
    # # i don't want this for part 2, and not neeed for part 1
    # if self.get_num_unpowered() == 0:
    #   print("error - no more unpowered boxes")
    #   return -1, -1
    
    best_dist = 1000 * 1000 * 1000 # check this later
    best_i = -1
    best_j = -1
    for i in range(self.num_boxes):
      # used to check if box was powered - but see now we should NOT do that
      for j in range(self.num_boxes):
        if i == j: 
          # skip if same box
          continue

        if self.are_boxes_directly_connected(i, j):
          # these two boxes are already directly conncected, so skip
          continue


        # can we use cached value
        if self.d_cache[i][j] > 0:
          d = self.d_cache[i][j]
        else:
          d = self.boxes[i].get_distance_from( self.boxes[j])
          self.d_cache[i][j] = d
        if d < best_dist:
          best_dist = d
          best_i = i
          best_j = j
          s = f"b{i:2d} b{j:2d} dist: {d:8.2f} is best min so far "
          if debug >= 2:
            print(s)

    if debug >= 1:
      print(f"Ret {best_i:2d} {self.boxes[best_i].stringify():15s} {best_j:2d} {self.boxes[best_j].stringify():15s} with {best_dist:12.1f} (unpowered: {self.get_num_unpowered()})")
    return best_i, best_j
    




  
  def connect_boxes_directly(self, i, j, debug = 0):
    # connect physically in cmap
    # and now both are powered on regardless of previous status
    self.boxes[i].set_to_powered()
    self.boxes[j].set_to_powered()
    self.cmap[i][j] = 1
    self.cmap[j][i] = 1

    if debug:
      print(f"directly connected {i} to {j}")
    
    return

  def on_same_logical_circuit(self, i, j):
    """
    Returns True if the logical circuit that contains i is the same as the logical circuit that contains j
    """
    # advance to name of LogicalCircuits 
    while self.status[i] == 0:
      i = self.parent[i] 
    while self.status[j] == 0:
      j = self.parent[j] 

    # at this point i and j are the names of their logical circuits
    return i == j


  def join_logical_circuits(self, i, j):
    if self.on_same_logical_circuit(i,j):
      #print(f"{i},{j} already the same logical circuit - no action")
      return

    # advance to name of LogicalCircuits 
    while self.status[i] == 0:
      i = self.parent[i] 
    while self.status[j] == 0:
      j = self.parent[j] 

    if i == j:
      # already on same logical circuit, but should have been caught earlier
      raise Exception("fix me")
    
    if i < j:
      self.members[i].append(j)
      for k in self.members[j]:
        self.members[i].append(k)
      self.members[i] = sorted(self.members[i]) # for clarity, not required
      self.status[j] = 0
      self.parent[j] = i
    elif i > j:
      self.members[j].append(i)
      for k in self.members[i]:
        self.members[j].append(k)
      self.members[j] = sorted(self.members[j]) # for clarity, not required
      self.status[i] = 0
      self.parent[i] = j
    else:
      raise Exception("fatal error - i==j?")
  
    return 
      
  def print_logical_circuits(self):
    for i in range(self.num_boxes):
      if self.status[i]:
        print(f"Circuit {i:2d} is {[i]} with {self.members[i]}")
      else:
        print(f"Circuit {i:2d} is invalid                                    (and points to {[i]})")



  def set_connections_to_stop_at(self, x = 10):
    if x <= 0 or x > 1000 * 1000:
      raise Exception("invalid request for con to stop at")
    self.term_condition = x
    print(f"Set connections_to_stop_at to {self.term_condition}")

  def set_to_run_until_single_circuit(self):
    self.term_condition = 0
  
  
  def process(self, debug = 0):
    """
    self.connections_to_stop_at = Part 1 says to stop at 10
    loops_to_stop_at - for debugging and handling runaway loops
    xor
    self.run_to_single_circuit
    """
    loops_run = 0
    loops_to_stop_at = 50000
    
    connections_made = 0    

    import time
    time_while = 0
    time_find = 0


    
    # if self.get_num_unpowered() == 0:
    #   if debug:
    #     print(f"process called but no unpowered boxes left")
    #     # make this an error, as should check earlier?
    #   return 
    while True:
      time_while_t0 = time.process_time()
              
      t0 = time.process_time()
      i, j = self.find_two_closest_unconnected_boxes(debug=0)

      self.last_i = i
      self.last_j = j
      
      time_find += time.process_time() - t0
      if debug:
        num_unpowered = self.get_num_unpowered()
        num_circuits = len(self.get_logical_circuits())
        s = f"Connecting: [{i:4d}:{j:4d}]    cons {connections_made:5d}  cis: {num_circuits:4d}  unp: {num_unpowered:4d}   time_find: {time_while:8.3f}  time_find: {time_while:8.3f}"
        if debug >= 2:
          print(s)
        elif debug == 1 and loops_run % 100 == 0:
          print(s)
        
      # if debug >= 1:
      #   num_unpowered = self.get_num_unpowered()
      #   num_circuits = len(self.get_logical_circuits())
      #   if loops_run % 100 == 0:
      #     print(f"Connecting: [{i:3d}:{j:3d}]      cons {connections_made:5d}  cis: {num_circuits:4d}  unp: {num_unpowered:4d}   time_find: {time_while:8.3f}  time_find: {time_while:8.3f}")
      #     #print(f"timeA: {timeA:5.1f}  timeB: {timeB:5.1f}  timeC: {timeC:5.1f}  timeD: {timeD:5.1f}  timeE: {timeE:5.1f}   timeF: {timeF:5.1f}")
      #     #print(timeA, timeB, timeC, timeD, timeE, timeF, timeG)
        

      
      self.connect_boxes_directly(i, j)
      
      self.join_logical_circuits(i,j)
      
      
      connections_made += 1
      # if debug > 1:
      #   print(f"                         ({self.get_num_unpowered()} unpowered after)")



      loops_run += 1

      if self.term_condition and connections_made == self.term_condition:
        print(f"Stopping as requested after making {connections_made} connections")
        break

      t0 = time.process_time()
      if self.term_condition == 0 and len(self.get_logical_circuits()) == 1:
        print(f"Stopping as only one logical circuit left")
        break
      
      # if self.get_num_unpowered() == 0:
      #   print(f"Stopping as no more unpowered.")
      #   break
          
      if loops_run == loops_to_stop_at:
        print(f"Stopping as ran for {loops_run:2} loops")
        break

      time_while += time.process_time() - time_while_t0

    # recap
    num_unpowered = self.get_num_unpowered()
    num_circuits = len(self.get_logical_circuits())
    print(f"Ending with:               cons {connections_made:5d}  cis: {num_circuits:4d}  unp: {num_unpowered:4d}   time_find: {time_while:8.3f}  time_find: {time_while:8.3f}")
    print(f"Ending with: {self.last_i:5d}  {self.last_j:5d}  cons {connections_made:5d}  cis: {num_circuits:4d}  unp: {num_unpowered:4d}   time_find: {time_while:8.3f}  time_find: {time_while:8.3f}")
    #last_x1 = self.boxes[self.last_i].getx()
    #last_x2 = self.boxes[self.last_j].getx()
    #print(f"{last_x1} * {last_x2} = {last_x1 * last_x2}")
    
    
    # return

  def print_last_x1x2_calc(self):
    print(f"Last connection was between {self.last_i} and {self.last_j}")
    x1 = self.boxes[self.last_i].getx()
    x2 = self.boxes[self.last_j].getx()
    print(f"{x1} * {x2} = {x1 * x2}")
  
  def get_logical_circuits(self):
    logical_circuits = []
    for i in range(self.num_boxes):
      if self.status[i] == 1:
        membership = [i]
        for i2 in self.members[i]:
          membership.append(i2)
        membership = sorted(membership)
        logical_circuits.append(membership)
    
    # sort lists by number of elements
    sorted_logical_circuits = sorted(logical_circuits, key=len, reverse = True)
    #sorted_logical_circuits = sorted( [len(x) for x in logical_circuits] )
    
    return sorted_logical_circuits
      
  def get_logical_circuit_calc(self):
    X = self.get_logical_circuits()
    return len(X[0]) * len(X[1]) * len(X[2])




In [852]:
day8_example = get_input_file("day8_example.txt")
P = Playground(day8_example)
P.set_connections_to_stop_at(10)
P.process(debug=2)
#P.print_box_connection_map()
#P.print_logical_circuits()
P.get_logical_circuit_calc()

Set connections_to_stop_at to 10
Connecting: [   0:  19]    cons     0  cis:   20  unp:   20   time_find:    0.000  time_find:    0.000
Connecting: [   0:   7]    cons     1  cis:   19  unp:   18   time_find:    0.000  time_find:    0.000
Connecting: [   2:  13]    cons     2  cis:   18  unp:   17   time_find:    0.000  time_find:    0.000
Connecting: [   7:  19]    cons     3  cis:   17  unp:   15   time_find:    0.001  time_find:    0.001
Connecting: [  17:  18]    cons     4  cis:   17  unp:   15   time_find:    0.001  time_find:    0.001
Connecting: [   9:  12]    cons     5  cis:   16  unp:   13   time_find:    0.001  time_find:    0.001
Connecting: [  11:  16]    cons     6  cis:   15  unp:   11   time_find:    0.001  time_find:    0.001
Connecting: [   2:   8]    cons     7  cis:   14  unp:    9   time_find:    0.001  time_find:    0.001
Connecting: [  14:  19]    cons     8  cis:   13  unp:    8   time_find:    0.001  time_find:    0.001
Connecting: [   2:  18]    cons     9  c

40

In [853]:
P = Playground(day8_example)
P.set_to_run_until_single_circuit()
P.process(debug=2)
P.print_last_x1x2_calc()

Connecting: [   0:  19]    cons     0  cis:   20  unp:   20   time_find:    0.000  time_find:    0.000
Connecting: [   0:   7]    cons     1  cis:   19  unp:   18   time_find:    0.000  time_find:    0.000
Connecting: [   2:  13]    cons     2  cis:   18  unp:   17   time_find:    0.001  time_find:    0.001
Connecting: [   7:  19]    cons     3  cis:   17  unp:   15   time_find:    0.001  time_find:    0.001
Connecting: [  17:  18]    cons     4  cis:   17  unp:   15   time_find:    0.001  time_find:    0.001
Connecting: [   9:  12]    cons     5  cis:   16  unp:   13   time_find:    0.001  time_find:    0.001
Connecting: [  11:  16]    cons     6  cis:   15  unp:   11   time_find:    0.001  time_find:    0.001
Connecting: [   2:   8]    cons     7  cis:   14  unp:    9   time_find:    0.001  time_find:    0.001
Connecting: [  14:  19]    cons     8  cis:   13  unp:    8   time_find:    0.001  time_find:    0.001
Connecting: [   2:  18]    cons     9  cis:   12  unp:    7   time_find: 

In [854]:
day8_input = get_input_file("day8_input.txt")
len(day8_input)

1000

In [855]:
P = Playground(day8_input)
P.set_to_run_until_single_circuit()
P.process(debug=1)

Connecting: [  78: 180]    cons     0  cis: 1000  unp: 1000   time_find:    0.000  time_find:    0.000
Connecting: [ 382: 891]    cons   100  cis:  906  unp:  826   time_find:    8.908  time_find:    8.908
Connecting: [ 541: 592]    cons   200  cis:  818  unp:  682   time_find:   17.498  time_find:   17.498
Connecting: [  73: 306]    cons   300  cis:  730  unp:  545   time_find:   26.156  time_find:   26.156
Connecting: [ 892: 936]    cons   400  cis:  649  unp:  438   time_find:   34.765  time_find:   34.765
Connecting: [ 151: 802]    cons   500  cis:  575  unp:  360   time_find:   43.385  time_find:   43.385
Connecting: [ 399: 977]    cons   600  cis:  505  unp:  300   time_find:   51.977  time_find:   51.977
Connecting: [   3: 112]    cons   700  cis:  447  unp:  253   time_find:   60.558  time_find:   60.558
Connecting: [ 284: 496]    cons   800  cis:  381  unp:  205   time_find:   69.156  time_find:   69.156
Connecting: [ 682: 951]    cons   900  cis:  323  unp:  165   time_find: 

In [856]:
P.print_last_x1x2_calc()

Last connection was between 792 and 847
30181 * 25594 = 772452514


### this runs crazy slow... 
idea: when connexting i and j, if they're already part of the same circuit I could skip making the physical connection
idea 2: could I skip making all physical connections?  why even track these?
better idea: this alg seems wrong - 8 minutes


In [858]:
# 772452514